In [ ]:
DROP TABLE IF EXISTS dbo.FactTaxiDaily;
DROP TABLE IF EXISTS dbo.FactAirQualityDaily;
DROP TABLE IF EXISTS dbo.FactCityHourly;
DROP TABLE IF EXISTS dbo.DimDate;
DROP TABLE IF EXISTS dbo.DimZone;
DROP TABLE IF EXISTS dbo.DimFX;
DROP TABLE IF EXISTS dbo.DimGDP;

In [ ]:
CREATE TABLE dbo.DimDate (
    date_key INT NOT NULL,
    date DATE NOT NULL,
    year INT NOT NULL,
    quarter INT NOT NULL,
    month INT NOT NULL,
    month_name VARCHAR(10) NOT NULL,
    day INT NOT NULL,
    day_of_week INT NOT NULL,
    day_name VARCHAR(10) NOT NULL,
    is_weekend BIT NOT NULL
);

INSERT INTO dbo.DimDate
SELECT
    YEAR(d.date) * 10000 + MONTH(d.date) * 100 + DAY(d.date) AS date_key,
    d.date,
    YEAR(d.date) AS year,
    DATEPART(QUARTER, d.date) AS quarter,
    MONTH(d.date) AS month,
    DATENAME(MONTH, d.date) AS month_name,
    DAY(d.date) AS day,
    DATEPART(WEEKDAY, d.date) AS day_of_week,
    DATENAME(WEEKDAY, d.date) AS day_name,
    CASE WHEN DATEPART(WEEKDAY, d.date) IN (1, 7) THEN 1 ELSE 0 END AS is_weekend
FROM (
    SELECT DATEADD(DAY, value, CAST('2023-01-01' AS DATE)) AS date
    FROM GENERATE_SERIES(0, 1095)
) AS d;

In [ ]:
CREATE TABLE dbo.DimZone (
    zone_key INT NOT NULL,
    borough VARCHAR(50) NOT NULL,
    zone VARCHAR(100) NOT NULL,
    service_zone VARCHAR(50) NOT NULL
);

INSERT INTO dbo.DimZone
SELECT
    z.location_id AS zone_key,
    z.borough,
    z.zone,
    z.service_zone
FROM silver.dbo.taxi_zones AS z;

In [ ]:
CREATE TABLE dbo.DimFX (
    date_key INT NOT NULL,
    usd_per_eur DECIMAL(10, 4) NOT NULL
);

INSERT INTO dbo.DimFX
SELECT
    YEAR(fx.date) * 10000 + MONTH(fx.date) * 100 + DAY(fx.date) AS date_key,
    CAST(fx.usd_per_eur AS DECIMAL(10, 4)) AS usd_per_eur
FROM silver.dbo.ecb_fx AS fx;

In [ ]:
CREATE TABLE dbo.DimGDP (
    year INT NOT NULL,
    country_code VARCHAR(3) NOT NULL,
    gdp_usd DECIMAL(20, 2) NOT NULL
);

INSERT INTO dbo.DimGDP
SELECT
    g.year,
    g.country_code,
    CAST(g.gdp_usd AS DECIMAL(20, 2)) AS gdp_usd
FROM silver.dbo.gdp AS g;

In [ ]:
CREATE TABLE dbo.FactAirQualityDaily (
    date_key INT NOT NULL,
    year INT NOT NULL,
    location_id INT NOT NULL,
    location_name VARCHAR(100) NOT NULL,
    parameter VARCHAR(10) NOT NULL,
    units VARCHAR(10) NOT NULL,
    avg_value FLOAT NOT NULL,
    min_value FLOAT NOT NULL,
    max_value FLOAT NOT NULL,
    measurement_count INT NOT NULL
);

INSERT INTO dbo.FactAirQualityDaily
SELECT
    YEAR(a.date) * 10000 + MONTH(a.date) * 100 + DAY(a.date) AS date_key,
    YEAR(a.date) AS year,
    a.location_id,
    a.location_name,
    a.parameter,
    a.units,
    AVG(a.value) AS avg_value,
    MIN(a.value) AS min_value,
    MAX(a.value) AS max_value,
    COUNT(*) AS measurement_count
FROM silver.dbo.air_quality AS a
GROUP BY a.date, a.location_id, a.location_name, a.parameter, a.units;

In [ ]:
CREATE TABLE dbo.FactTaxiDaily (
    date_key INT NOT NULL,
    zone_key INT NOT NULL,
    year INT NOT NULL,
    trip_count INT NOT NULL,
    total_fare_usd DECIMAL(18, 2) NOT NULL,
    total_amount_usd DECIMAL(18, 2) NOT NULL,
    total_distance_miles DECIMAL(18, 2) NOT NULL,
    total_duration_min DECIMAL(18, 2) NOT NULL,
    avg_fare_usd DECIMAL(10, 2) NOT NULL,
    avg_amount_usd DECIMAL(10, 2) NOT NULL,
    avg_distance_miles DECIMAL(10, 2) NOT NULL,
    avg_duration_min DECIMAL(10, 2) NOT NULL
);

INSERT INTO dbo.FactTaxiDaily
SELECT
    YEAR(t.pickup_date) * 10000 + MONTH(t.pickup_date) * 100 + DAY(t.pickup_date) AS date_key,
    t.pu_location_id AS zone_key,
    YEAR(t.pickup_date) AS year,
    COUNT(*) AS trip_count,
    CAST(SUM(t.fare_amount) AS DECIMAL(18, 2)) AS total_fare_usd,
    CAST(SUM(t.total_amount) AS DECIMAL(18, 2)) AS total_amount_usd,
    CAST(SUM(t.trip_distance) AS DECIMAL(18, 2)) AS total_distance_miles,
    CAST(SUM(t.trip_duration_min) AS DECIMAL(18, 2)) AS total_duration_min,
    CAST(AVG(t.fare_amount) AS DECIMAL(10, 2)) AS avg_fare_usd,
    CAST(AVG(t.total_amount) AS DECIMAL(10, 2)) AS avg_amount_usd,
    CAST(AVG(t.trip_distance) AS DECIMAL(10, 2)) AS avg_distance_miles,
    CAST(AVG(t.trip_duration_min) AS DECIMAL(10, 2)) AS avg_duration_min
FROM silver.dbo.nyc_taxi_trips AS t
GROUP BY t.pickup_date, t.pu_location_id;

In [ ]:
CREATE TABLE dbo.FactCityHourly (
    date_key INT NOT NULL,
    hour INT NOT NULL,
    trips INT NOT NULL,
    pm25_avg FLOAT NULL,
    no2_avg FLOAT NULL
);

INSERT INTO dbo.FactCityHourly
SELECT
    t.date_key,
    t.hour,
    t.trips,
    a.pm25_avg,
    a.no2_avg
FROM (
    SELECT
        YEAR(pickup_date) * 10000 + MONTH(pickup_date) * 100 + DAY(pickup_date) AS date_key,
        pickup_hour AS hour,
        COUNT(*) AS trips
    FROM silver.dbo.nyc_taxi_trips
    GROUP BY pickup_date, pickup_hour
) AS t
LEFT JOIN (
    SELECT
        YEAR(date) * 10000 + MONTH(date) * 100 + DAY(date) AS date_key,
        hour,
        AVG(CASE WHEN parameter = 'pm25' THEN value END) AS pm25_avg,
        AVG(CASE WHEN parameter = 'no2' THEN value END) AS no2_avg
    FROM silver.dbo.air_quality
    GROUP BY date, hour
) AS a
    ON a.date_key = t.date_key
    AND a.hour = t.hour;

In [ ]:
ALTER TABLE dbo.DimDate ADD CONSTRAINT PK_DimDate PRIMARY KEY NONCLUSTERED (date_key) NOT ENFORCED;
ALTER TABLE dbo.DimZone ADD CONSTRAINT PK_DimZone PRIMARY KEY NONCLUSTERED (zone_key) NOT ENFORCED;
ALTER TABLE dbo.DimFX ADD CONSTRAINT PK_DimFX PRIMARY KEY NONCLUSTERED (date_key) NOT ENFORCED;
ALTER TABLE dbo.DimGDP ADD CONSTRAINT PK_DimGDP PRIMARY KEY NONCLUSTERED (year) NOT ENFORCED;
ALTER TABLE dbo.FactTaxiDaily ADD CONSTRAINT PK_FactTaxiDaily PRIMARY KEY NONCLUSTERED (date_key, zone_key) NOT ENFORCED;
ALTER TABLE dbo.FactAirQualityDaily ADD CONSTRAINT PK_FactAirQualityDaily PRIMARY KEY NONCLUSTERED (date_key, location_id, parameter) NOT ENFORCED;
ALTER TABLE dbo.FactCityHourly ADD CONSTRAINT PK_FactCityHourly PRIMARY KEY NONCLUSTERED (date_key, hour) NOT ENFORCED;

In [ ]:
ALTER TABLE dbo.FactTaxiDaily ADD CONSTRAINT FK_FactTaxiDaily_DimDate FOREIGN KEY (date_key) REFERENCES dbo.DimDate (date_key) NOT ENFORCED;
ALTER TABLE dbo.FactTaxiDaily ADD CONSTRAINT FK_FactTaxiDaily_DimFX FOREIGN KEY (date_key) REFERENCES dbo.DimFX (date_key) NOT ENFORCED;
ALTER TABLE dbo.FactTaxiDaily ADD CONSTRAINT FK_FactTaxiDaily_DimZone FOREIGN KEY (zone_key) REFERENCES dbo.DimZone (zone_key) NOT ENFORCED;
ALTER TABLE dbo.FactTaxiDaily ADD CONSTRAINT FK_FactTaxiDaily_DimGDP FOREIGN KEY (year) REFERENCES dbo.DimGDP (year) NOT ENFORCED;
ALTER TABLE dbo.FactAirQualityDaily ADD CONSTRAINT FK_FactAirQualityDaily_DimDate FOREIGN KEY (date_key) REFERENCES dbo.DimDate (date_key) NOT ENFORCED;
ALTER TABLE dbo.FactAirQualityDaily ADD CONSTRAINT FK_FactAirQualityDaily_DimGDP FOREIGN KEY (year) REFERENCES dbo.DimGDP (year) NOT ENFORCED;
ALTER TABLE dbo.FactCityHourly ADD CONSTRAINT FK_FactCityHourly_DimDate FOREIGN KEY (date_key) REFERENCES dbo.DimDate (date_key) NOT ENFORCED;

In [ ]:
SELECT 'DimDate' AS table_name, COUNT(*) AS row_count FROM dbo.DimDate
UNION ALL
SELECT 'DimZone', COUNT(*) FROM dbo.DimZone
UNION ALL
SELECT 'DimFX', COUNT(*) FROM dbo.DimFX
UNION ALL
SELECT 'DimGDP', COUNT(*) FROM dbo.DimGDP
UNION ALL
SELECT 'FactTaxiDaily', COUNT(*) FROM dbo.FactTaxiDaily
UNION ALL
SELECT 'FactAirQualityDaily', COUNT(*) FROM dbo.FactAirQualityDaily
UNION ALL
SELECT 'FactCityHourly', COUNT(*) FROM dbo.FactCityHourly;

In [ ]:
SELECT 'FactTaxiDaily -> DimDate' AS link, COUNT(*) AS rows_without_pair
FROM dbo.FactTaxiDaily AS f
LEFT JOIN dbo.DimDate AS d ON d.date_key = f.date_key
WHERE d.date_key IS NULL
UNION ALL
SELECT 'FactTaxiDaily -> DimFX', COUNT(*)
FROM dbo.FactTaxiDaily AS f
LEFT JOIN dbo.DimFX AS fx ON fx.date_key = f.date_key
WHERE fx.date_key IS NULL
UNION ALL
SELECT 'FactTaxiDaily -> DimZone', COUNT(*)
FROM dbo.FactTaxiDaily AS f
LEFT JOIN dbo.DimZone AS z ON z.zone_key = f.zone_key
WHERE z.zone_key IS NULL
UNION ALL
SELECT 'FactTaxiDaily -> DimGDP', COUNT(*)
FROM dbo.FactTaxiDaily AS f
LEFT JOIN dbo.DimGDP AS g ON g.year = f.year
WHERE g.year IS NULL
UNION ALL
SELECT 'FactAirQualityDaily -> DimDate', COUNT(*)
FROM dbo.FactAirQualityDaily AS a
LEFT JOIN dbo.DimDate AS d ON d.date_key = a.date_key
WHERE d.date_key IS NULL
UNION ALL
SELECT 'FactAirQualityDaily -> DimGDP', COUNT(*)
FROM dbo.FactAirQualityDaily AS a
LEFT JOIN dbo.DimGDP AS g ON g.year = a.year
WHERE g.year IS NULL
UNION ALL
SELECT 'FactCityHourly -> DimDate', COUNT(*)
FROM dbo.FactCityHourly AS h
LEFT JOIN dbo.DimDate AS d ON d.date_key = h.date_key
WHERE d.date_key IS NULL;

In [ ]:
SELECT
    SUM(CASE WHEN date_key IS NULL THEN 1 ELSE 0 END) AS null_date_key,
    SUM(CASE WHEN zone_key IS NULL THEN 1 ELSE 0 END) AS null_zone_key,
    SUM(CASE WHEN trip_count IS NULL THEN 1 ELSE 0 END) AS null_trip_count,
    SUM(CASE WHEN total_amount_usd IS NULL THEN 1 ELSE 0 END) AS null_total_amount,
    SUM(CASE WHEN avg_fare_usd IS NULL THEN 1 ELSE 0 END) AS null_avg_fare
FROM dbo.FactTaxiDaily;

SELECT
    SUM(CASE WHEN date_key IS NULL THEN 1 ELSE 0 END) AS null_date_key,
    SUM(CASE WHEN location_id IS NULL THEN 1 ELSE 0 END) AS null_location_id,
    SUM(CASE WHEN parameter IS NULL THEN 1 ELSE 0 END) AS null_parameter,
    SUM(CASE WHEN avg_value IS NULL THEN 1 ELSE 0 END) AS null_avg_value
FROM dbo.FactAirQualityDaily;

SELECT
    SUM(CASE WHEN date_key IS NULL THEN 1 ELSE 0 END) AS null_date_key,
    SUM(CASE WHEN hour IS NULL THEN 1 ELSE 0 END) AS null_hour,
    SUM(CASE WHEN trips IS NULL THEN 1 ELSE 0 END) AS null_trips,
    SUM(CASE WHEN pm25_avg IS NULL THEN 1 ELSE 0 END) AS null_pm25_avg,
    SUM(CASE WHEN no2_avg IS NULL THEN 1 ELSE 0 END) AS null_no2_avg
FROM dbo.FactCityHourly;

In [ ]:
SELECT SUM(CAST(trip_count AS BIGINT)) AS trips FROM dbo.FactTaxiDaily;

SELECT SUM(measurement_count) AS measurements FROM dbo.FactAirQualityDaily;

SELECT SUM(CAST(trips AS BIGINT)) AS trips FROM dbo.FactCityHourly;

SELECT TOP 5 * FROM dbo.FactCityHourly ORDER BY date_key, hour;